# 🔑 Solutions: Lesson 12 · Project 4 · LSTM Maize Price Forecasting

> **Instructor's note.** Don't read these until you've really tried! For each problem: read the statement, look at the solution, then *close the notebook and re-implement it from memory* in the lesson notebook. The self-checks below are identical to the ones you saw there.

Each solution cell is self-contained and runs top-to-bottom.

In [1]:
import sys; sys.path.append("..")
import numpy as np
import matplotlib.pyplot as plt
from utils import dlcourse as dl
import pandas as pd, math
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
    cpu = torch.device("cpu")
except ImportError:
    pass
dl.set_seed(42)
%matplotlib inline

---
### 🧩 Problem 12.1: Chronological split  &nbsp; `⭐`

Write `chronological_masks(dates, train_end, val_end)` returning three boolean arrays `(train, val, test)` over a `DatetimeIndex`: train is `dates <= train_end`, val is `train_end < dates <= val_end`, test is after `val_end`. They must be disjoint and cover everything.


In [2]:
def chronological_masks(dates, train_end, val_end):
    te, ve = pd.Timestamp(train_end), pd.Timestamp(val_end)
    return np.asarray(dates <= te), np.asarray((dates > te) & (dates <= ve)), np.asarray(dates > ve)

In [3]:
# ▶ self-check
d = pd.date_range("2020-01-01", periods=10, freq="W-MON")
r = chronological_masks(d, "2020-02-01", "2020-02-29")
dl.check("returns three masks", r is not None and len(r) == 3)
dl.check("masks are disjoint and cover all weeks", r is not None and (r[0].astype(int) + r[1] + r[2] == 1).all())
dl.check("order is train → val → test in time", r is not None and r[0].sum() == 4 and r[1].sum() == 4 and r[2].sum() == 2)
dl.check("no test date is earlier than any train date", r is not None and d[r[2]].min() > d[r[0]].max())

✅ returns three masks
✅ masks are disjoint and cover all weeks
✅ order is train → val → test in time
✅ no test date is earlier than any train date


---
### 🧩 Problem 12.2: Forecast metrics  &nbsp; `⭐⭐`

Write `forecast_metrics(actual, pred, naive_pred)` returning a dict with `MAE`, `RMSE`, `MAPE` (in %), and `rel_MAE` = MAE of `pred` ÷ MAE of `naive_pred`. All inputs are 1-D arrays.


In [4]:
def forecast_metrics(actual, pred, naive_pred):
    err = pred - actual
    mae = np.abs(err).mean()
    return {"MAE": mae, "RMSE": np.sqrt((err ** 2).mean()), "MAPE": 100 * np.mean(np.abs(err) / np.abs(actual)),
            "rel_MAE": mae / np.abs(naive_pred - actual).mean()}

In [5]:
# ▶ self-check
a = np.array([100., 200., 400.]); p = np.array([110., 190., 380.]); nv = np.array([100., 100., 100.])
m = forecast_metrics(a, p, nv)
dl.check("MAE = 40/3", m is not None and np.isclose(m["MAE"], 40 / 3))
dl.check("RMSE = sqrt(200)", m is not None and np.isclose(m["RMSE"], np.sqrt(200)))
dl.check("MAPE = (10 + 5 + 5)/3 %", m is not None and np.isclose(m["MAPE"], 20 / 3))
dl.check("rel_MAE = (40/3)/(400/3) = 0.1", m is not None and np.isclose(m["rel_MAE"], 0.1))

✅ MAE = 40/3
✅ RMSE = sqrt(200)
✅ MAPE = (10 + 5 + 5)/3 %
✅ rel_MAE = (40/3)/(400/3) = 0.1


---
### 🧩 Problem 12.3: Seasonal-naive baseline  &nbsp; `⭐⭐`

Write `seasonal_naive(logp, origins, horizon=4, season=52)`. For each origin `t` it predicts the log-change over the next `horizon` weeks to equal **what happened one season earlier**:
`logp[t+1-season : t+horizon+1-season] − logp[t-season]`. Return an array of shape `(len(origins), horizon)`.


In [6]:
def seasonal_naive(logp, origins, horizon=4, season=52):
    return np.stack([logp[t + 1 - season:t + horizon + 1 - season] - logp[t - season] for t in origins])

In [7]:
# ▶ self-check
logp = np.log(100 + np.arange(120) + 5 * np.sin(np.arange(120) * 2 * np.pi / 52))
out = seasonal_naive(logp, np.array([60, 70]), horizon=4, season=52)
dl.check("shape (2, 4)", out is not None and out.shape == (2, 4))
dl.check("first row = last year's change", out is not None and np.allclose(out[0], logp[9:13] - logp[8]))
dl.check("second row", out is not None and np.allclose(out[1], logp[19:23] - logp[18]))

✅ shape (2, 4)
✅ first row = last year's change
✅ second row


---
### 🧩 Problem 12.4: Prediction-interval coverage  &nbsp; `⭐⭐`

Write `interval_from_residuals(val_residuals, level=0.8)` returning `(q_lo, q_hi)`: the `(1−level)/2` and `(1+level)/2` quantiles of each **column** of the validation residuals:
and `coverage(y, pred, q_lo, q_hi)` returning, per column, the fraction of `y` inside `[pred+q_lo, pred+q_hi]`.


In [8]:
def interval_from_residuals(val_residuals, level=0.8):
    a = (1 - level) / 2
    return np.quantile(val_residuals, a, axis=0), np.quantile(val_residuals, 1 - a, axis=0)

def coverage(y, pred, q_lo, q_hi):
    return ((y >= pred + q_lo) & (y <= pred + q_hi)).mean(axis=0)

In [9]:
# ▶ self-check
rng = np.random.default_rng(0)
res = rng.normal(0, 1, (5000, 2)) * [1, 3]
r = interval_from_residuals(res, 0.8)
dl.check("returns two arrays of length 2", r is not None and len(r) == 2 and len(r[0]) == 2)
dl.check("wider spread → wider interval (col 1 ≈ 3× col 0)", r is not None and np.isclose((r[1] - r[0])[1] / (r[1] - r[0])[0], 3, atol=0.3))
y = rng.normal(0, 1, (5000, 2)) * [1, 3]; pred = np.zeros((5000, 2))
c = coverage(y, pred, *r) if r is not None else None
dl.check("coverage ≈ 0.8 on fresh data from the same distribution", c is not None and np.allclose(c, 0.8, atol=0.03))

✅ returns two arrays of length 2
✅ wider spread → wider interval (col 1 ≈ 3× col 0)
✅ coverage ≈ 0.8 on fresh data from the same distribution


---
### 🧩 Problem 12.5: Plug the scaling leak  &nbsp; `⭐`

A colleague standardised the *whole* series before splitting. Write `scale_split(train, test)` that computes mean/std from **`train` only** and returns `(train_scaled, test_scaled)`. For a rising series the scaled test values should lie *outside* the train range: as they honestly would in production.


In [10]:
def scale_split(train, test):
    mu, sd = train.mean(), train.std()
    return (train - mu) / sd, (test - mu) / sd

In [11]:
# ▶ self-check
series_ = np.linspace(100, 200, 100); tr, te = series_[:70], series_[70:]
r = scale_split(tr, te)
dl.check("train is standardised (mean 0, std 1)", r is not None and np.isclose(r[0].mean(), 0, atol=1e-9) and np.isclose(r[0].std(), 1))
dl.check("test uses TRAIN statistics (so it's > train's max)", r is not None and r[1].min() > r[0].max())

✅ train is standardised (mean 0, std 1)
✅ test uses TRAIN statistics (so it's > train's max)


**Why this works.** If scaling used the whole series, the test data would look comfortably in-range and hide the extrapolation problem you will face in deployment.

---
### 🧩 Problem 12.6: Back-test a hold-or-sell rule  &nbsp; `⭐⭐⭐`

Write `holding_gain(p_now, p_future, hold, cost=0.03)`. `hold` is a boolean array (the rule's decision per week). Return `(avg_gain, fraction_held)` where, for held weeks, gain = `p_future − p_now·(1+cost)`, and for weeks we sell immediately gain = 0.
Then write `decide(p_now, forecast, cost=0.03)` returning the hold mask: hold when `forecast/p_now − 1 > cost`.


In [12]:
def holding_gain(p_now, p_future, hold, cost=0.03):
    gain = np.where(hold, p_future - p_now * (1 + cost), 0.0)
    return gain.mean(), hold.mean()

def decide(p_now, forecast, cost=0.03):
    return (forecast / p_now - 1) > cost

In [13]:
# ▶ self-check
p0 = np.array([100., 100., 100., 100.]); pf = np.array([110., 100., 120., 90.])
h = decide(p0, np.array([106., 101., 130., 95.]))
dl.check("decide → [True, False, True, False]", h is not None and list(h) == [True, False, True, False])
r = holding_gain(p0, pf, h) if h is not None else None
dl.check("avg gain = ((110−103) + (120−103)) / 4 = 6.0", r is not None and np.isclose(r[0], 6.0))
dl.check("fraction held = 0.5", r is not None and np.isclose(r[1], 0.5))

✅ decide → [True, False, True, False]
✅ avg gain = ((110−103) + (120−103)) / 4 = 6.0
✅ fraction held = 0.5


**Why this works.** Counting the 'sell now' weeks as zero keeps strategies comparable: always-hold, never-hold and rule-based all average over the same set of decisions.